# 📖 Notebook 1 — Why Service Discovery?

In a microservices world, services come and go all the time:

- They **scale up** when traffic spikes, **scale down** when it dies off.
- They **crash** and get restarted on a different machine with a different IP.
- A new version rolls out; old instances disappear.

So the question is simple but thorny: **how does service A find a healthy instance of service B _right now_?**

That is exactly what **service discovery** solves. Think of it as a *runtime phone book*:
services **register** when they start, and other services **look them up** by name.

This notebook walks the classic **bad → better → best** progression:

1. ❌ **Bad**: hard-code the address in code/config.
2. ⚠️ **Better**: an in-memory registry — but dead addresses linger when instances crash.
3. ✅ **Best**: registry + **heartbeats with a TTL** so dead instances are forgotten automatically.


## 🛠️ Setup

```bash
cd 05-microservices/service-discovery
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear,
reload the window: `Cmd+Shift+P` → **Reload Window**.

> All examples below are **pure Python, no extra dependencies** — we simulate network calls
> with plain function calls so you can focus on the pattern, not the plumbing.


## ❌ Bad: hard-coded addresses

The "just put the URL in a config file" approach. Works on day one, falls over on day two.


In [ ]:
# The classic anti-pattern: every caller knows the exact host:port of a dependency.

ORDERS_URL = "http://10.0.0.1:8080"   # baked in at deploy time

def call_orders(order_id):
    # Pretend this is requests.get(f"{ORDERS_URL}/orders/{order_id}")
    print(f"GET {ORDERS_URL}/orders/{order_id}")
    return {"status": 200, "order_id": order_id}

call_orders(42)


### Why this breaks in production

- You want **2 instances** of `orders` for redundancy → the URL can only point to one.
- Instance `10.0.0.1` crashes and Kubernetes spawns a replacement at `10.0.0.7` → every caller
  is now pointing at a dead box until someone ships a config change.
- You want to scale 2 → 10 during Black Friday → no way to express that here.

The real world needs a *dynamic* answer, not a string constant.


## ⚠️ Better: an in-memory service registry

Let's introduce a tiny **Service Registry** — the "phone book" at the heart of the pattern.
Instances register their address under a logical service name; clients look them up by name.


In [ ]:
class NaiveRegistry:
    """service_name -> set of (host, port) tuples."""
    def __init__(self):
        self.services = {}

    def register(self, name, host, port):
        self.services.setdefault(name, set()).add((host, port))
        print(f"registered {name} @ {host}:{port}")

    def deregister(self, name, host, port):
        self.services.get(name, set()).discard((host, port))

    def lookup(self, name):
        return list(self.services.get(name, set()))


reg = NaiveRegistry()
reg.register("orders", "10.0.0.1", 8080)
reg.register("orders", "10.0.0.2", 8080)
print("instances of orders:", reg.lookup("orders"))


### The hidden bug: nothing removes dead instances

What happens if `10.0.0.2` **crashes** before it can call `deregister`?


In [ ]:
# Simulate a crash: the process dies instantly and can't deregister.
# The registry happily keeps handing out the dead address.

import random

DEAD = {"10.0.0.2"}          # this box died a second ago; nobody told the registry

def http_get(host, port, path):
    """Pretend HTTP call. A dead box refuses the connection."""
    if host in DEAD:
        raise ConnectionError(f"connection refused: {host}:{port}")
    return {"status": 200}

def naive_client_call(registry, service, path):
    instances = sorted(registry.lookup(service))     # sorted → reproducible demo
    if not instances:
        raise RuntimeError(f"no instances for {service}")
    host, port = random.choice(instances)
    return http_get(host, port, path)

random.seed(0)
ok = failed = 0
for _ in range(20):
    try:
        naive_client_call(reg, "orders", "/v1/42")
        ok += 1
    except ConnectionError:
        failed += 1

print(f"registry still lists: {sorted(reg.lookup('orders'))}")
print(f"20 requests -> {ok} succeeded, {failed} FAILED with connection refused")
print()
print(f"That's a {failed/20:.0%} error rate that will last forever, because nothing")
print("in this design ever notices 10.0.0.2 is gone. Scaling to 10 instances doesn't")
print("help either — it just changes the fraction.")


Half our traffic is failing and **the error rate never recovers**. Deregistration is a
best-effort courtesy that a crashed process cannot perform — and those are exactly the
processes you most need removed. Retries don't save you either: a retry picks from the
same stale list and has the same odds of landing on the corpse.

So the registry has to *prove* liveness continuously rather than *assume* it.

## ✅ Best: heartbeats + TTL

The classic solution: instances periodically send a **heartbeat** ("I'm still alive!") to the
registry. Each heartbeat refreshes a timestamp. On lookup, the registry returns only instances
whose last heartbeat is within a **TTL** (time-to-live) window. Crashed instances stop heart-beating
and silently fall off the list.

This is how **Consul, Eureka, and most service meshes** model liveness.


In [ ]:
import time

class Registry:
    """Maps service_name -> {(host, port): last_heartbeat_timestamp}."""
    def __init__(self, ttl=5.0):
        self.services = {}
        self.ttl = ttl  # seconds

    def register(self, name, host, port):
        self.services.setdefault(name, {})[(host, port)] = time.time()

    def heartbeat(self, name, host, port):
        # A healthy instance calls this on a timer (e.g. every 1s).
        if (host, port) in self.services.get(name, {}):
            self.services[name][(host, port)] = time.time()

    def deregister(self, name, host, port):
        self.services.get(name, {}).pop((host, port), None)

    def healthy_instances(self, name):
        now = time.time()
        return [addr for addr, hb in self.services.get(name, {}).items()
                if now - hb < self.ttl]

    def reap(self, name, expire_after=None):
        """Actually delete long-dead entries.

        `healthy_instances` only *filters* stale rows — they stay in the dict forever,
        which is a slow memory leak on a fleet that autoscales all day. Real registries
        evict after a grace period well beyond the TTL (Consul's
        `deregister_critical_service_after`, Eureka's eviction task)."""
        expire_after = self.ttl * 10 if expire_after is None else expire_after
        now = time.time()
        entries = self.services.get(name, {})
        for addr in [a for a, hb in entries.items() if now - hb > expire_after]:
            del entries[addr]
            print(f"reaped {name} @ {addr[0]}:{addr[1]} (silent for >{expire_after:.0f}s)")


r = Registry(ttl=2.0)
r.register("orders", "10.0.0.1", 8080)
r.register("orders", "10.0.0.2", 8080)
print("healthy now:", r.healthy_instances("orders"))


### Watch the crash recover itself

We'll pretend instance `10.0.0.2` crashed — it simply stops heart-beating.
Only `10.0.0.1` keeps sending heartbeats. Within one TTL window `10.0.0.2` disappears from
the list — **no human intervention required**.


In [ ]:
# Instance 1 keeps heart-beating. Instance 2 is dead (no more heartbeats).
# Wait long enough for instance 2's entry to go stale.
time.sleep(2.1)
r.heartbeat("orders", "10.0.0.1", 8080)

print("healthy after crash + TTL:", r.healthy_instances("orders"))
print("  → 10.0.0.2 silently drops out of every lookup. Traffic stops hitting it.")

# It is filtered out, but still sitting in the dict. Eventually, evict it.
print("still stored:", sorted(r.services["orders"].keys()))
r.reap("orders", expire_after=2.0)
print("after reaping:", sorted(r.services["orders"].keys()))


### Client-side discovery: the client picks an instance itself

With a healthy list in hand, the client chooses one and calls it directly. We use
`random.choice` for brevity; in practice you'd use round-robin, least-loaded, or
consistent-hashing strategies.


In [ ]:
def client_side_call(registry, service, path):
    instances = registry.healthy_instances(service)
    if not instances:
        raise RuntimeError(f"no healthy instances for {service}")
    host, port = random.choice(instances)
    print(f"calling http://{host}:{port}{path}")
    return {"status": 200}

client_side_call(r, "orders", "/v1/orders/42")


### ⏱️ Picking the heartbeat interval and TTL

Two knobs you'll always tune together:

- **Heartbeat interval** — how often each instance pings the registry.
- **TTL** — how long the registry trusts the last heartbeat.

Rules of thumb used by Eureka/Consul-shaped systems:

- Set `TTL ≈ 2–3 × heartbeat interval`. One lost packet shouldn't kill you.
- Short TTL (e.g. 2 s) → dead instances disappear fast, but more registry traffic.
- Long TTL (e.g. 30 s) → cheaper, but callers hit dead boxes for longer.

A common production starting point: **heartbeat every 10 s, TTL 30 s**. Tune from
real data, not vibes.


## 🧠 What to remember

| Stage | What changes | Why it matters |
|-------|--------------|----------------|
| ❌ Hard-coded URL | Address is a constant | One instance, zero elasticity, breaks on restart |
| ⚠️ Plain registry | Addresses looked up by name | Supports scaling, but stale entries linger on crashes |
| ✅ Registry + heartbeat + TTL | Liveness is continuously proved | Dead instances disappear automatically |

Next, **Notebook 2** goes deeper: **server-side** vs **client-side** discovery,
**self-registration** vs **third-party registration**, **push heartbeats** vs
**pull health checks**, and client-side **caching** for when the registry itself is down.
